
# Theory Notebook: Linear Regression, Random Forests, XGBoost, MAE, and RMSE

This notebook is a **theory-focused companion** for practical machine learning work.  
It explains:

- **Linear Regression**
- **Random Forests**
- **XGBoost**
- **MAE**
- **RMSE**

It also includes:
- intuitive explanations
- mathematical background
- visuals
- runnable code examples
- parameter explanations
- guidance on when to use each method

---

## Learning goals

By the end of this notebook, you should be able to:

1. explain how linear regression works
2. explain how tree-based models differ from linear models
3. understand how Random Forests reduce overfitting
4. understand the boosting idea behind XGBoost
5. compute and interpret MAE and RMSE
6. choose a reasonable model for a regression task


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor, plot_importance
    HAS_XGB = True
except Exception:
    HAS_XGB = False

np.random.seed(42)



# 1. Regression in one sentence

A **regression model** predicts a **continuous numeric value**.

Examples:
- house price
- room temperature
- CO₂ concentration
- power consumption
- rainfall amount

This differs from **classification**, where the output is a category such as:
- spam / not spam
- heating / cooling / idle
- fraud / not fraud

---

## A tiny synthetic example

We start with a simple dataset to visualize model behavior.


In [ ]:
X, y = make_regression(
    n_samples=220,
    n_features=1,
    noise=18,
    random_state=42
)

# Make it easier to visualize
X = np.sort(X[:, 0]).reshape(-1, 1)
y = y[np.argsort(X[:, 0])]

plt.figure(figsize=(10, 4))
plt.scatter(X, y, s=18)
plt.title("Simple 1D regression dataset")
plt.xlabel("Input x")
plt.ylabel("Target y")
plt.grid(alpha=0.3)
plt.show()



# 2. Linear Regression

## Core idea

Linear regression assumes the target can be approximated by a **linear combination of features**:

$$
\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p
$$

where:

- $\hat{y}$ is the prediction
- $\beta_0$ is the intercept
- $\beta_1, \beta_2, \dots, \beta_p$ are coefficients
- $x_1, x_2, \dots, x_p$ are input features

---

## Intuition

Linear regression tries to fit the **best straight line** or, in higher dimensions, the best **hyperplane**.

### What does "best" mean?
Usually: the model minimizes the **sum of squared errors**:

$$
\text{SSE} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

This means large errors are punished strongly.

---

## What the coefficients mean

If all other variables are held fixed:

- a **positive coefficient** means the prediction increases as the feature increases
- a **negative coefficient** means the prediction decreases as the feature increases
- a coefficient near **zero** means the feature has little linear effect

This interpretability is one of the biggest strengths of linear regression.


In [ ]:
lin = LinearRegression()
lin.fit(X, y)
y_pred_lin = lin.predict(X)

plt.figure(figsize=(10, 4))
plt.scatter(X, y, s=18, label="Data")
plt.plot(X, y_pred_lin, linewidth=2, label="Linear Regression")
plt.title("Linear Regression fit")
plt.xlabel("Input x")
plt.ylabel("Target y")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Intercept:", lin.intercept_)
print("Coefficient:", lin.coef_[0])



## Why linear regression is useful

### Advantages
- simple
- fast
- interpretable
- good baseline
- works well when relationships are close to linear

### Limitations
- struggles with strongly nonlinear relationships
- sensitive to outliers
- may underfit complex patterns

---

## Example API usage

In scikit-learn:

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression(
    fit_intercept=True,
    positive=False
)

model.fit(X_train, y_train)
pred = model.predict(X_test)
```

### Important parameters

#### `fit_intercept`
- `True`: learn an intercept term
- `False`: assume the line should pass through the origin

Usually keep this as `True` unless you have a strong reason not to.

#### `positive`
- `True`: forces coefficients to be non-negative
- `False`: coefficients can be positive or negative

This can be useful in constrained domains, but it is not common as a default choice.



# 3. From one tree to many trees

Before Random Forests and XGBoost, it helps to understand a **decision tree**.

A regression tree works by repeatedly asking questions like:

- is temperature < 20?
- is power > 50?
- is motion detected?

Each split divides the data into smaller groups.  
The final prediction is often the **average target value** in a leaf.

Trees can model nonlinear behavior much better than linear regression.


In [ ]:
# Nonlinear toy data
x_curve = np.linspace(-3, 3, 250)
y_curve = 2 * np.sin(2 * x_curve) + 0.4 * x_curve**2 + np.random.normal(0, 0.35, len(x_curve))
X_curve = x_curve.reshape(-1, 1)

tree = DecisionTreeRegressor(max_depth=3, random_state=42)
tree.fit(X_curve, y_curve)
y_tree = tree.predict(X_curve)

plt.figure(figsize=(10, 4))
plt.scatter(X_curve, y_curve, s=12, label="Data")
plt.plot(X_curve, y_tree, linewidth=2, label="Decision Tree (depth=3)")
plt.title("Decision tree regression creates piecewise-constant predictions")
plt.xlabel("Input x")
plt.ylabel("Target y")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(tree, filled=True, rounded=True, feature_names=["x"])
plt.title("Example regression tree")
plt.show()



## Why trees are powerful

Trees:
- capture nonlinear relationships
- naturally model interactions between variables
- work with mixed patterns better than a simple line

But a single tree has a problem:

### It can overfit
If the tree becomes too deep, it can memorize noise in the training set.

This is where **Random Forests** and **XGBoost** become important.



# 4. Random Forests

## Core idea

A Random Forest is an **ensemble of many decision trees**.

Each tree is trained on a slightly different version of the data:
- using a **bootstrap sample** of the training set
- using a random subset of features at each split

The final prediction is the **average** of all trees.

$$
\hat{y}_{forest} = \frac{1}{T}\sum_{t=1}^{T} \hat{y}^{(t)}
$$

where $T$ is the number of trees.

---

## Intuition

A single tree may be unstable:
small changes in the training data can produce a very different tree.

A forest reduces this instability by averaging many trees.

### Idea in plain language
- one tree may be wrong in one region
- another tree may be wrong in another region
- averaging many trees reduces variance

This is why Random Forests are usually much more robust than a single tree.


In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=6,
    random_state=42
)
rf.fit(X_curve, y_curve)
y_rf = rf.predict(X_curve)

plt.figure(figsize=(10, 4))
plt.scatter(X_curve, y_curve, s=12, label="Data")
plt.plot(X_curve, y_tree, linewidth=2, label="Single Tree")
plt.plot(X_curve, y_rf, linewidth=2, label="Random Forest")
plt.title("Random Forest gives smoother and more stable predictions")
plt.xlabel("Input x")
plt.ylabel("Target y")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



## Why Random Forests often work well

### Advantages
- strong default model
- captures nonlinearities
- handles feature interactions
- less sensitive to scaling
- often works well with little tuning
- provides feature importance estimates

### Limitations
- less interpretable than linear regression
- can be slower than linear models
- large forests can use more memory
- cannot extrapolate well outside the training range

---

## Important parameters

Example:

```python
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
```

### `n_estimators`
Number of trees in the forest.

- larger values usually improve stability
- but training becomes slower

Common values: 100, 200, 500

### `max_depth`
Maximum depth of each tree.

- small depth: simpler trees, lower variance, more bias
- large depth: more flexible trees, higher variance

### `min_samples_split`
Minimum number of samples required to split a node.

Larger values make the model more conservative.

### `min_samples_leaf`
Minimum number of samples in each leaf.

Higher values smooth predictions and reduce overfitting.

### `max_features`
Number of features considered at each split.

This randomness helps make trees different from one another.

### `random_state`
Used for reproducibility.

### `n_jobs`
Number of CPU cores to use.
- `-1` means: use all available cores


In [ ]:
# Random Forest feature importance example on a multi-feature dataset
X_multi, y_multi = make_regression(
    n_samples=500,
    n_features=6,
    n_informative=4,
    noise=20,
    random_state=42
)
feature_names = [f"x{i}" for i in range(X_multi.shape[1])]

rf_multi = RandomForestRegressor(n_estimators=200, random_state=42)
rf_multi.fit(X_multi, y_multi)

importance = pd.Series(rf_multi.feature_importances_, index=feature_names).sort_values(ascending=False)

importance.plot(kind="bar", figsize=(8, 4), title="Random Forest feature importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

importance



# 5. XGBoost

## What does the name mean?

**XGBoost** stands for **Extreme Gradient Boosting**.

It is a very popular implementation of **gradient boosted trees**.

---

## Core idea

Unlike Random Forests, where trees are built **independently** and then averaged, boosting builds trees **sequentially**.

Each new tree tries to correct the errors made by the previous trees.

### Conceptually
1. start with a simple prediction
2. compute the errors
3. train a new tree to model those errors
4. add that correction to the model
5. repeat many times

So the final model is:

$$
\hat{y} = \sum_{m=1}^{M} f_m(x)
$$

where each $f_m$ is a small decision tree.

---

## Intuition

Random Forest:
- many trees vote independently

XGBoost:
- each tree learns from the mistakes of earlier trees

This often makes XGBoost:
- more accurate
- more tuneable
- more sensitive to hyperparameters



## Why XGBoost is popular

### Strengths
- often excellent predictive performance
- handles nonlinearities and interactions
- supports regularization
- can model complex structured tabular data
- often performs very well in competitions and real-world tasks

### Trade-offs
- more hyperparameters
- easier to overfit if tuned badly
- less interpretable than linear regression
- sometimes less beginner-friendly than Random Forests


In [ ]:
if HAS_XGB:
    xgb = XGBRegressor(
        n_estimators=180,
        max_depth=3,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42
    )
    xgb.fit(X_curve, y_curve)
    y_xgb = xgb.predict(X_curve)

    plt.figure(figsize=(10, 4))
    plt.scatter(X_curve, y_curve, s=12, label="Data")
    plt.plot(X_curve, y_rf, linewidth=2, label="Random Forest")
    plt.plot(X_curve, y_xgb, linewidth=2, label="XGBoost")
    plt.title("XGBoost can fit nonlinear structure very effectively")
    plt.xlabel("Input x")
    plt.ylabel("Target y")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("XGBoost is not installed in this environment. The theory below still applies.")



## Important XGBoost parameters

Example:

```python
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42
)
```

### `n_estimators`
Number of boosting rounds, often described as the number of trees.

- more trees can improve fit
- too many trees can overfit
- lower learning rates often require more trees

### `learning_rate`
How much each new tree contributes.

- small value: slower learning, often more stable
- large value: faster learning, but higher overfitting risk

Common pattern:
- lower `learning_rate`
- higher `n_estimators`

### `max_depth`
Maximum depth of each tree.

- deeper trees can model more complex patterns
- but deeper trees also increase overfitting risk

### `subsample`
Fraction of training samples used for each tree.

- values below 1.0 add randomness
- often helps generalization

### `colsample_bytree`
Fraction of features used when growing each tree.

Also helps regularization.

### `reg_alpha`
L1 regularization term.

Encourages sparsity or simpler models.

### `reg_lambda`
L2 regularization term.

Helps control model complexity.

### `objective`
For regression, usually:
- `"reg:squarederror"`

### `random_state`
Controls reproducibility.



# 6. MAE and RMSE

When we build a regression model, we need a way to evaluate how good it is.

Two common metrics are:

- **MAE**: Mean Absolute Error
- **RMSE**: Root Mean Squared Error

---

## MAE

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|
$$

This is the average absolute difference between prediction and truth.

### Interpretation
If MAE = 12, then the model is wrong by about 12 units on average.

### Strength
Easy to explain.

### Behavior
All errors contribute linearly.

---

## RMSE

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}
$$

This squares errors before averaging, then takes the square root.

### Interpretation
Still in the same unit as the target, but it penalizes large errors more strongly.

### Strength
Useful when large mistakes are especially bad.

---

## Key difference

- **MAE** treats a 10-unit error as twice as bad as a 5-unit error
- **RMSE** punishes large errors more heavily because of squaring

So RMSE is usually more sensitive to outliers or rare bad predictions.


In [ ]:
true = np.array([100, 110, 120, 130, 140])
pred_good = np.array([102, 108, 121, 128, 141])
pred_bad_outlier = np.array([102, 108, 121, 128, 170])

def metric_table(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"Prediction set": name, "MAE": mae, "RMSE": rmse}

table = pd.DataFrame([
    metric_table(true, pred_good, "Mostly small errors"),
    metric_table(true, pred_bad_outlier, "One large error"),
])
table


In [ ]:
errors_good = np.abs(true - pred_good)
errors_bad = np.abs(true - pred_bad_outlier)

x = np.arange(len(true))
width = 0.35

plt.figure(figsize=(10, 4))
plt.bar(x - width/2, errors_good, width=width, label="Mostly small errors")
plt.bar(x + width/2, errors_bad, width=width, label="One large error")
plt.xticks(x, [f"Sample {i}" for i in range(len(true))])
plt.ylabel("Absolute error")
plt.title("Why RMSE reacts strongly to large errors")
plt.legend()
plt.tight_layout()
plt.show()



## Metric API usage

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred)
```

### Why `squared=False`?
`mean_squared_error(...)` normally returns **MSE**.  
Setting `squared=False` returns **RMSE** instead.

---

## When to prefer which?

### Prefer MAE when:
- interpretability matters
- you want the average absolute error
- large errors should not dominate too much

### Prefer RMSE when:
- large errors are especially costly
- you want to punish rare bad predictions more strongly

In practice, many people report **both**.



# 7. Side-by-side comparison

Now we compare the three models on the same nonlinear dataset.


In [ ]:
# Train/test split for comparison
idx = int(0.75 * len(X_curve))
X_train, X_test = X_curve[:idx], X_curve[idx:]
y_train, y_test = y_curve[:idx], y_curve[idx:]

lin2 = LinearRegression()
lin2.fit(X_train, y_train)
pred_lin = lin2.predict(X_test)

rf2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    random_state=42
)
rf2.fit(X_train, y_train)
pred_rf = rf2.predict(X_test)

rows = []
rows.append({
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(y_test, pred_lin),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred_lin))
})
rows.append({
    "Model": "Random Forest",
    "MAE": mean_absolute_error(y_test, pred_rf),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred_rf))
})

pred_xgb = None
if HAS_XGB:
    xgb2 = XGBRegressor(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42
    )
    xgb2.fit(X_train, y_train)
    pred_xgb = xgb2.predict(X_test)
    rows.append({
        "Model": "XGBoost",
        "MAE": mean_absolute_error(y_test, pred_xgb),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred_xgb))
    })

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
results


In [ ]:
plt.figure(figsize=(11, 5))
plt.scatter(X_test, y_test, s=18, label="True test data")
plt.plot(X_test, pred_lin, linewidth=2, label="Linear Regression")
plt.plot(X_test, pred_rf, linewidth=2, label="Random Forest")
if pred_xgb is not None:
    plt.plot(X_test, pred_xgb, linewidth=2, label="XGBoost")

plt.title("Model predictions on the same test data")
plt.xlabel("Input x")
plt.ylabel("Target y")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
results.plot(
    x="Model",
    y=["MAE", "RMSE"],
    kind="bar",
    figsize=(9, 4),
    title="Metric comparison"
)
plt.ylabel("Error")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()



# 8. How to choose between them

## Choose Linear Regression when:
- you want a simple baseline
- interpretability matters
- relationships are approximately linear
- speed is important
- the number of features is not huge and the structure is simple

## Choose Random Forest when:
- the problem is nonlinear
- you want a strong default model
- you want relatively low tuning effort
- feature interactions matter
- you want a balance between performance and simplicity

## Choose XGBoost when:
- you want very strong predictive performance
- you are comfortable tuning hyperparameters
- the dataset is structured/tabular
- you can spend more effort optimizing the model

---

## Rule of thumb for teaching
A very practical sequence is:

1. start with **Linear Regression**
2. compare with **Random Forest**
3. then introduce **XGBoost** as a stronger boosted-tree approach



# 9. Common mistakes

## Linear Regression
- assuming linearity when the pattern is nonlinear
- ignoring outliers
- misinterpreting correlation as causation

## Random Forest
- using too few trees
- assuming feature importance means causal importance
- forgetting that forests do not extrapolate well

## XGBoost
- setting a high learning rate with many deep trees
- overfitting through aggressive tuning
- comparing models without a proper validation split

## Metrics
- reporting only one metric
- ignoring the target unit
- using random train/test splits for time series



# 10. Minimal end-to-end examples

## Linear Regression

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression(fit_intercept=True)
model.fit(X_train, y_train)
pred = model.predict(X_test)
```

## Random Forest

```python
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
```

## XGBoost

```python
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
```

## Metrics

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print("MAE:", mae)
print("RMSE:", rmse)
```



# 11. Final summary

## Linear Regression
- simple
- interpretable
- best for roughly linear relationships
- excellent baseline

## Random Forest
- ensemble of many trees
- robust and strong default for nonlinear tabular problems
- easy to use effectively

## XGBoost
- boosted trees trained sequentially
- often very strong performance
- more hyperparameter-sensitive

## MAE
- average absolute error
- easy to interpret

## RMSE
- square-rooted average squared error
- penalizes large errors more strongly

---

## Practical recommendation

For a new regression problem:
1. build a **Linear Regression** baseline
2. compare with **Random Forest**
3. try **XGBoost** if you want stronger performance
4. report both **MAE** and **RMSE**



# 12. Optional exercises

1. Create a dataset where linear regression performs better than trees.
2. Create a dataset where trees clearly outperform linear regression.
3. Change `max_depth` in Random Forest and observe the effect.
4. Change `learning_rate` and `n_estimators` in XGBoost and compare results.
5. Construct a case where MAE and RMSE lead to different conclusions.
